In [1]:
import torch
from torch import nn
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from model.encoder_block import Encoder_Block

In [2]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size: int, context_length: int, model_dim: int, num_blocks: int, num_heads: int):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, model_dim)
        self.pos_embedding = nn.Embedding(context_length, model_dim)
        self.transformer_blocks = nn.Sequential()
        for _ in range(num_blocks):
            self.transformer_blocks.append(Encoder_Block(model_dim, num_heads))
        self.layer_norm_three = nn.LayerNorm(model_dim)
        self.vocab_projection = nn.Linear(model_dim, vocab_size)

    def forward(self, context):
        embedded = self.token_embedding(context)
        context_length = context.shape[1]
        positions = torch.arange(context_length).to(device)
        embedded = embedded + self.pos_embedding(positions)

        raw_output = self.vocab_projection(self.layer_norm_three(self.transformer_blocks(embedded)))
        # raw_output is BxTxV, where V is the vocabulary size
        return raw_output

In [ ]:
vocab_size = 5 # THe number of different tokens the model recognizes
context_length = 5 # How many tokens back the model can read
model_dim = 16 # Feature dimensionality for embeddings and attention
num_blocks = 4 # Number of repetitions of Transformer block
num_heads = 4 # Number of self attention instances

encoder = Encoder(vocab_size, context_length, model_dim, num_blocks, num_heads)
encoder = encoder.to(device)

context = [['With', 'great', 'power', 'comes', 'great']] # BxT
mapping = {'with': 0, 'great': 1, 'power': 2, 'comes': 3, 'responsibility': 4}
context_indices = [[mapping[token.lower()] for token in seq] for seq in context]


context = torch.tensor(context_indices, dtype=torch.long).to(device)

probabilities = encoder(context)
print(probabilities)

tensor([[[ 0.4150,  0.5142, -1.1863,  0.5922, -0.0387],
         [-0.0865,  0.6608, -0.4606,  0.1558, -0.2343],
         [ 0.1272,  0.3792, -0.1142, -0.5131,  0.4139],
         [-0.1685,  0.6652,  1.4025,  0.3321, -0.3729],
         [ 0.4537,  0.1952,  0.8750,  1.0858, -0.1154]]], device='cuda:0',
       grad_fn=<ViewBackward0>)
